# Forecast metrics — benchmarking SMARD's day-ahead forecasts

Implements [`.claude/specs/04-forecast-metrics.md`](../../.claude/specs/04-forecast-metrics.md).

SMARD publishes a day-ahead forecast of residual load (`fc_residual_load`) and of its two parts
(`fc_grid_load`, `fc_gen_wind_solar`). That public forecast is the **baseline our own model has to
beat**. This notebook measures how good it is.

## What this notebook produces

- The **hourly error series** `error = forecast − actual` for three pairs:
  - residual load: `fc_residual_load` ↔ `residual_load`
  - grid load: `fc_grid_load` ↔ `grid_load`
  - wind + solar: `fc_gen_wind_solar` ↔ `renewables`
- **MAE, RMSE, bias and `hour_count`** per pair, plus nMAE (normalised MAE) and a capacity-normalised generation error
- slices by **time** (month, hour of day, season, year), by **residual-load level**, and the error of the **day maximum / minimum**
- a **cascade** of error tables (full record → trailing 365 days → year → month → day), all from one `window_metrics` helper
- three exported files in `data/metrics/`:
  - `smard_forecast_errors_hourly.csv`: the source of truth for re-scoring SMARD on any window
  - `smard_forecast_errors_daily.csv`
  - `smard_benchmark_metrics.csv`

## Conventions

- **Sign:** `error = forecast − actual`. **Positive error / positive bias = over-forecast**, negative = under-forecast.
- **Every metric is an average over a window of hours** and is always shown with its `hour_count`. The headline numbers are provisional: the modelling spec re-scores SMARD on its own test window.
- **Units:** hourly readings and hourly errors in `MWh`; normalised errors in `%`; timing offsets in hours (`h`); installed capacity in `MW`. An hourly `MWh` reading equals the mean power over that hour, so `err_renewables / cap_wind_solar` reads as a share of installed capacity.
- `time_series` is the shared hourly frame, holding exactly `SERIES + DERIVED`. The errors live in a separate `errors` frame.
- No literal calendar year or date appears in code. Everything year-dependent derives from `YEARS` or from the data.
- **Durations, never row counts.**

## Not in this notebook

- anything using the risk-label files (error on risk hours, flag agreement): parked in [04.3](../../.claude/specs/04.3-risk-label-link.md)
- naive baselines and skill scores: parked in [04.1](../../.claude/specs/04.1-naive-baseline.md)
- ISO-week level, zoom demos, tolerance share: parked in [04.2](../../.claude/specs/04.2-streamlit-views.md)
- MAPE, any model of our own, the train/test split, MLflow, reBAP

## 1 Setup

Same setup as [`team-EDA.ipynb`](../01_eda/team-EDA.ipynb) §1, as reused by
[`risk-definition.ipynb`](../03_risk_classification/risk-definition.ipynb) §1. Inherited, not re-derived:

- the data-directory resolver (walks **upward** from the working directory, so the notebook runs from any folder in the repo)
- loading, renaming, the German-CSV float conversion and the dtype asserts
- `time_series`, `SERIES`, `DERIVED`, `YEARS`, `DAY_NAMES`, the season mapping
- `_complete_periods`, `period_mean`, `period_energy`, `style_timeseries`, `seasonal_plot`
- the duration constants and the day-completeness rule from `risk-definition.ipynb` §3.1

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it with
`notebooks/API-connection.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"Data directory: {DATA}")

### 1.1 Helpers

Unchanged from `team-EDA.ipynb`:

- `_complete_periods` drops calendar periods the data does not fully cover. A plain `.resample()` produces fake edge dips.
- `style_timeseries` requires an explicit `ylabel`, so no plot can ship without stating its unit.

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    A period counts only if it starts not earlier than the first observation and ends not later
    than the last observation's closing edge. Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (
        periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(series, freq):
    """Mean of `series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts.
    """
    agg = series.groupby(series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(series, freq, drop_incomplete=True):
    """Per-period aggregate of `series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view.
        Deliberately not ``mwh_per_day / 24``: a month containing the spring DST switch holds
        743 hours, not 744.
    ``hours``, ``days``
        the two denominators.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`.
    """
    grouped = series.groupby(series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")


def seasonal_plot(df, y_value, title, ylabel):
    """Creates a seasonal plot from a dataframe.

    Args:
        df (DataFrame): frame with separate `month` and `year` columns, already aggregated to
            one row per (year, month). Passing raw hourly data makes seaborn bootstrap a
            confidence interval per cell over tens of thousands of rows.
        y_value (str): name of the y-value to plot
        title (str): title of the plot
        ylabel (str): axis description, including units. Required, and actually applied.
    """
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=df,
        x="month",
        y=y_value,
        hue="year",
        palette="viridis",
        legend=True,
        ax=ax
    )
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.show()

### 1.2 Load and prepare

- The dtype assertion guards against the German Excel-CSV conversion silently leaving a column as text.
- Everything after this cell uses `time_series`.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar"
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)
time_series.head(3)

### 1.3 Engineered columns

- `YEARS` is computed from the loaded data and is the only permitted source of year information.
- `spans_gap` marks the row *following* a gap: at each spring DST switch the local hour **02:00 does not exist**.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
    "cap_wind_off", "cap_wind_on", "cap_solar"
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.4 Rule constants, expressed as durations

Inherited from `risk-definition.ipynb` §3.1. Every rule about a length of time is written as a
**duration** and converted to an observation count from the **measured** resolution:

- the trailing window is **365 days**, not 8,760 rows
- a day is **complete** if it carries at least **23/24** of its expected observations. This accepts the 23-hour spring-DST day and rejects materially short days.

A switch to SMARD's quarter-hour data would therefore change the observation counts but none of the definitions.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

WINDOW = pd.Timedelta(days=365)      # trailing window: headline table and rolling lines
DAY_COMPLETENESS = 23 / 24           # accepts the spring-DST day, rejects materially short days

EXPECTED_OBS_PER_DAY = int(pd.Timedelta("1D") / RESOLUTION)
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

# One row per local calendar date — the same day boundary DERIVED["date"] uses, not a rolling 24h.
DAY = pd.Series(time_series.index.normalize(), index=time_series.index)
DAYS = pd.DatetimeIndex(sorted(DAY.unique()))

print(f"resolution        : {RESOLUTION}  ->  {EXPECTED_OBS_PER_DAY} observations per full day")
print(f"trailing window   : {WINDOW.days} days")
print(f"day completeness  : >= {MIN_OBS_PER_DAY} of {EXPECTED_OBS_PER_DAY} observations")
print(f"calendar days     : {len(DAYS):,}  ({DAYS.min():%Y-%m-%d} .. {DAYS.max():%Y-%m-%d})")

### 1.5 Initial self-check

Structural only, and deliberately free of any hardcoded row count or date bound: the record's
extent is expected to change. The closing self-check (§9) re-runs these invariants plus a comparison
against `LOADED`.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

**What we are working with**

- **One hourly record, 67,336 rows (2019-01-01 00:00 → 2026-09-06 23:00), 2,806 calendar days.**
  - The final year is **partial** (it ends in September). Every year-on-year statement below therefore comes from full 365-day windows, never from a partial year against a full one.
  - The extent is a snapshot and moves with every re-fetch. Nothing below depends on these numbers being fixed.

- **Every forecast/actual pair already sits side by side in the first row.**
  - At 2019-01-01 00:00 SMARD forecast `fc_residual_load` = 19,264.5 MWh against an actual `residual_load` of 19,825.5 MWh. That is an error of **−561 MWh**: an under-forecast under this notebook's sign convention.
  - §2 turns this into an error series for all three pairs and every hour.

---

## 2 Errors and their decomposition

The errors live in a **separate `errors` frame** on `time_series.index`, not as new columns on
`time_series`, so `time_series` keeps exactly `SERIES + DERIVED`.

Per pair the frame holds the actual, the forecast and `err_{actual} = forecast − actual`. These are
the same pair columns the hourly export writes in §8:

| Pair | Forecast | Actual | Error column |
|---|---|---|---|
| residual load | `fc_residual_load` | `residual_load` | `err_residual_load` |
| grid load | `fc_grid_load` | `grid_load` | `err_grid_load` |
| wind + solar | `fc_gen_wind_solar` | `renewables` | `err_renewables` |

It also carries the installed wind + solar capacity
`cap_wind_solar = cap_wind_off + cap_wind_on + cap_solar` (MW), so `window_metrics` (§3) can compute
nMAE and the capacity-normalised generation error from this one frame.

**Sign convention: positive error / positive bias = over-forecast, negative = under-forecast.**

In [ ]:
# One entry per forecast/actual pair, keyed by the actual column. Every later section loops over this.
PAIRS = {
    "residual_load": {"forecast": "fc_residual_load", "label": "Residual load", "color": "#1C1C1C"},
    "grid_load": {"forecast": "fc_grid_load", "label": "Grid load", "color": "#2C6EBA"},
    "renewables": {"forecast": "fc_gen_wind_solar", "label": "Wind + solar", "color": "#2E8B57"},
}
ERR = {actual: f"err_{actual}" for actual in PAIRS}

errors = pd.DataFrame(index=time_series.index)
for actual, spec in PAIRS.items():
    errors[actual] = time_series[actual]
    errors[spec["forecast"]] = time_series[spec["forecast"]]
    # NaN wherever either side is missing, so the pairwise-complete rule (2.2) is built in.
    errors[ERR[actual]] = time_series[spec["forecast"]] - time_series[actual]

errors["cap_wind_solar"] = time_series[["cap_wind_off", "cap_wind_on", "cap_solar"]].sum(axis=1)

assert errors.index.equals(time_series.index)
assert list(time_series.columns) == SERIES + DERIVED, "time_series must not gain error columns"

print(f"errors: {errors.shape[0]:,} rows x {errors.shape[1]} columns")
print(f"columns: {list(errors.columns)}")
errors.head(3)

### 2.1 The forecast-side identity and the error decomposition

`risk-definition.ipynb` §1.7 showed that the actual residual load is load minus wind and solar:
`residual_load = grid_load − renewables`. If SMARD builds its forecast the same way,

$$\text{fc\_residual\_load} = \text{fc\_grid\_load} - \text{fc\_gen\_wind\_solar},$$

then subtracting the two identities gives an exact decomposition of the error:

$$\text{err\_residual\_load} = \text{err\_grid\_load} - \text{err\_renewables}.$$

Both identities are checked below. The decomposition can deviate by at most the sum of the two
identity deviations, and that bound is the tolerance the closing self-check (§9) uses.

In [ ]:
def identity_deviation(total, load, generation):
    """|total − (load − generation)| on the hours where all three exist."""
    return (total - (load - generation)).abs().dropna()


fc_dev = identity_deviation(
    time_series["fc_residual_load"], time_series["fc_grid_load"], time_series["fc_gen_wind_solar"]
)
act_dev = identity_deviation(
    time_series["residual_load"], time_series["grid_load"], time_series["renewables"]
)
decomp_dev = identity_deviation(
    errors["err_residual_load"], errors["err_grid_load"], errors["err_renewables"]
)

# Algebraic bound: the decomposition deviation is the forecast-side minus the actual-side deviation.
DECOMP_TOL = fc_dev.max() + act_dev.max()

for name, dev in [("forecast identity", fc_dev), ("actual identity", act_dev),
                  ("error decomposition", decomp_dev)]:
    over_1 = dev > 1
    line = (f"{name:20s}: {len(dev):,} hours, exact {(dev == 0).mean():6.2%}, "
            f"<= 1 MWh {(dev <= 1).mean():7.3%}, max {dev.max():,.2f} MWh")
    if over_1.any():
        line += (f"  |  {int(over_1.sum())} hours > 1 MWh, "
                 f"{dev[over_1].index.min():%Y-%m-%d} .. {dev[over_1].index.max():%Y-%m-%d}")
    print(line)

print(f"\ndecomposition tolerance DECOMP_TOL = {DECOMP_TOL:,.2f} MWh "
      f"(max forecast-side + max actual-side deviation)")
assert decomp_dev.max() <= DECOMP_TOL + 1e-9

# A worked example from the data: the hour with the largest residual-load error.
worst = errors["err_residual_load"].abs().idxmax()
e = errors.loc[worst]
print(f"\nlargest |err_residual_load| at {worst}:")
print(f"  err_grid_load     {e['err_grid_load']:>10,.0f} MWh")
print(f"  err_renewables    {e['err_renewables']:>10,.0f} MWh")
print(f"  err_residual_load {e['err_residual_load']:>10,.0f} MWh "
      f"(= {e['err_grid_load']:,.0f} − ({e['err_renewables']:,.0f}))")

**Both identities hold, so every residual-load error splits exactly into a load part and a generation part.**

- **SMARD builds its residual-load forecast as load minus wind + solar.**
  - The forecast identity holds in all 67,312 hours with a forecast, to at most 0.01 MWh of rounding.

- **The decomposition `err_residual_load = err_grid_load − err_renewables` is exact to within 16.5 MWh.**
  - 99.9 % of hours are within 1 MWh. The 57 exceptions all fall in 2021-01-04 .. 2021-01-06 and come entirely from the *actual*-side identity, a SMARD artefact already seen in `risk-definition.ipynb` §1.7.
  - The largest of them, 16.5 MWh, is negligible against the error sizes below.

- **An under-forecast of wind + solar shows up as an over-forecast of residual load, and the other way round.**
  - The largest residual-load error in the record, at 2026-04-05 13:00, is **−27,047 MWh**. It is almost entirely generation: wind + solar was forecast at ≈ 74,500 MWh but ≈ 44,900 MWh was recorded (`err_renewables` = +29,599 MWh), while the load error was only +2,551 MWh.
  - The actual residual load stayed just below zero all midday while the forecast expected ≈ −29,000 MWh. This is consistent with curtailment of wind and solar, which `data/smard.csv` cannot confirm.

### 2.2 Pairwise-complete hours

An hour enters a pair's metric **only if both forecast and actual exist**. Missing values are never
interpolated. The `err_*` columns are already `NaN` wherever either side is missing, so every
metric built on them follows this rule automatically. `hour_count` is always the number of hours
actually compared, per pair and per window.

Two cases produce missing hours: a forecast SMARD never published, and, after a re-fetch, the latest
forecasts whose actuals are not published yet. Both are counted from the data below, never
hardcoded.

In [ ]:
rows = []
for actual, spec in PAIRS.items():
    err = errors[ERR[actual]]
    missing = err.isna()
    missing_days = sorted({d.strftime("%Y-%m-%d") for d in err.index[missing].normalize()})
    rows.append({
        "pair": spec["label"],
        "hour_count": int(err.notna().sum()),
        "missing forecast": int(errors[spec["forecast"]].isna().sum()),
        "missing actual": int(errors[actual].isna().sum()),
        "days with a gap": ", ".join(missing_days) if missing_days else "—",
    })

pairwise = pd.DataFrame(rows).set_index("pair")

# The last hour at which all three pairs are observed: the anchor for every trailing window (§3).
JOINT_END = errors[list(ERR.values())].dropna().index.max()

print(f"rows in time_series          : {len(time_series):,}")
print(f"last jointly observed hour   : {JOINT_END}  (last row: {time_series.index.max()})")
pairwise

**Missing data is one day and affects two of the three pairs.**

- **Residual load and grid load are compared on 67,312 hours, wind + solar on all 67,336.**
  - The gap is the 24 hours of 2020-01-31, where SMARD published no grid-load forecast and therefore no residual-load forecast. Under the pairwise rule those hours drop out of those two pairs only.

- **The record currently ends on a jointly observed hour.**
  - The last row (2026-09-06 23:00) has all three forecasts and actuals, so the trailing 365-day window in §3 ends there. After a re-fetch that carries forecasts ahead of actuals, `JOINT_END` would move earlier on its own.

---

## 3 Metrics

Every number in this notebook is an average over a **window of hours**. Per pair and window:

| Metric | Definition | Unit |
|---|---|---|
| **MAE** | mean absolute error: mean of `abs(error)` | MWh |
| **RMSE** | square root of the mean of `error²` large misses are penalized more heavily than MAE. | MWh |
| **bias** | mean signed error; positive = over-forecast | MWh |
| **`hour_count`** | number of hours actually compared (pairwise-complete, §2.2) | hours |
| **nMAE** | normalised MAE: `MAE / mean(actual)` over the same hours, so the three pairs are comparable | % |
| **capacity-normalised MAE / bias** | mean of `abs(err_renewables) / cap_wind_solar` and of `err_renewables / cap_wind_solar` — wind + solar only | % of installed capacity |

**One function computes all of them.** `window_metrics(errors, start, end, mask)` returns the
table above for any window or boolean hour mask, and `metrics_by(errors, key)` applies it per group
(month, hour of day, season, year, day).

Every table in this notebook comes from these two
functions, so all metrics share one definition. The spec numbers this helper B23 (the cascade), but
it is defined here because every slice before the cascade needs it.

**RMSE is always recomputed from the squared hourly errors of the window.** It is never averaged
across sub-windows: the mean of twelve monthly RMSEs is not the yearly RMSE.

### 3.1 Where nMAE and the capacity-normalised error are reported

**nMAE** is reported only at the **full-record, trailing-365-day, year, season and month** levels,
and as a rolling 365-day line (§4). It is **never** reported for **residual load at day level or in
level bins**. Residual load's mean over a short window can come close to zero, and the ratio
explodes there. The cell below shows how close it gets in this record. The hour-of-day slice (§4)
reports MAE and bias only.

The **capacity-normalised error** is reported for wind + solar only, at **year level** and as the
**rolling 365-day line**, and never per month. The `cap_*` columns are a yearly step value, so
within-year fleet growth is not visible, and only a full-year window matches the step. This measure
answers "did the wind + solar forecast get better?" across years: the absolute generation error grows
with the fleet even if the forecast does not get worse.

`window_metrics` enforces both rules through two switches: `nmae` (on by default) and `capacity`
(off by default).

In [ ]:
# How close does residual load's window mean get to zero? Complete days only (the §1.4 rule).
res = time_series["residual_load"]
day_obs = res.groupby(DAY).size()
daily_mean = res.groupby(DAY).mean()[day_obs >= MIN_OBS_PER_DAY]
monthly_mean = period_mean(res, "M")

print(f"residual load, whole-record mean : {res.mean():>9,.0f} MWh")
print(f"lowest monthly mean              : {monthly_mean.min():>9,.0f} MWh  ({monthly_mean.idxmin():%Y-%m})")
print(f"lowest daily mean                : {daily_mean.min():>9,.0f} MWh  ({daily_mean.idxmin():%Y-%m-%d})")
print(f"days with a mean below 5,000 MWh : {int((daily_mean < 5000).sum())} of {len(daily_mean):,}")

# Daily means against individual hours, per year: midday lows are averaged against the night.
day_min = res.groupby(DAY).min()[day_obs >= MIN_OBS_PER_DAY]
low_days = pd.DataFrame({
    "complete days": daily_mean.groupby(daily_mean.index.year).size(),
    "daily mean < 5,000 MWh": (daily_mean < 5000).groupby(daily_mean.index.year).sum(),
    "any hour < 5,000 MWh": (day_min < 5000).groupby(day_min.index.year).sum(),
    "any hour < 0 MWh": (day_min < 0).groupby(day_min.index.year).sum(),
}).rename_axis("year")
low_days

**The nMAE exclusion is needed: a daily residual-load mean can be close to zero.**

- **Monthly means are safe, daily means are not.**
  - The lowest monthly mean is 19,746 MWh (2025-06), well away from zero, so monthly nMAE stays meaningful.
  - The lowest daily mean is only **679 MWh** (2025-10-26). The full-record MAE of ≈ 2,600 MWh (§3.3) divided by that would be an nMAE of ≈ 380 %, which is a statement about the denominator, not about the forecast.
  - Only 8 of 2,806 days have a **daily mean** below 5,000 MWh, but 7 of them fall in 2025–2026
    - Individual hours go far lower: in 2026 alone, 101 days already carry at least one negative hour
    - As the solar fleet grows, more daily means will approach zero, which is why the rule is a fixed exclusion rather than a threshold.

### 3.2 `window_metrics` and `metrics_by`

- `start` and `end` are **inclusive** hour timestamps. `None` means the start or end of the record.
- `mask` is a boolean Series on `errors.index` (or an array of the same length). It selects hours inside the window.
- The result has one row per pair, indexed by the pair key (`residual_load`, `grid_load`, `renewables`). Columns that a switch turns off stay `NaN`.
- `show_metrics` only formats a table for display: pair labels, units in the column names, and columns that are empty everywhere are dropped.

In [ ]:
METRICS = ["MAE", "RMSE", "bias", "nMAE_pct", "cap_MAE_pct", "cap_bias_pct", "hour_count"]
PAIR_LABEL = {actual: spec["label"] for actual, spec in PAIRS.items()}


def window_metrics(errors, start=None, end=None, mask=None, nmae=True, capacity=False):
    """MAE, RMSE, bias and hour_count per pair over one window of hours.

    Pairwise-complete: an hour counts for a pair only where its `err_*` is not NaN. RMSE is
    computed from this window's squared hourly errors, never from sub-window RMSEs.

    nmae      include nMAE = MAE / mean(actual) in %. Callers switch it off at day level, in
              level bins and in the hour-of-day slice (Behaviour 7).
    capacity  include the capacity-normalised MAE / bias of wind + solar in % of installed
              capacity. Callers switch it on only for full-year windows (Behaviour 8).
    """
    frame = errors.loc[start:end]
    if mask is not None:
        if not isinstance(mask, pd.Series):
            mask = pd.Series(np.asarray(mask), index=errors.index)
        frame = frame[mask.reindex(frame.index, fill_value=False).astype(bool)]

    rows = {}
    for actual in PAIRS:
        err = frame[ERR[actual]]
        compared = err.notna()
        e = err[compared]
        row = dict.fromkeys(METRICS, np.nan)
        row["hour_count"] = int(compared.sum())
        if row["hour_count"]:
            row["MAE"] = e.abs().mean()
            row["RMSE"] = np.sqrt((e ** 2).mean())
            row["bias"] = e.mean()
            if nmae:
                row["nMAE_pct"] = 100 * row["MAE"] / frame.loc[compared, actual].mean()
            if capacity and actual == "renewables":
                share = e / frame.loc[compared, "cap_wind_solar"]
                row["cap_MAE_pct"] = 100 * share.abs().mean()
                row["cap_bias_pct"] = 100 * share.mean()
        rows[actual] = row

    table = pd.DataFrame.from_dict(rows, orient="index")[METRICS]
    table["hour_count"] = table["hour_count"].astype(int)
    return table.rename_axis("pair")


def metrics_by(errors, key, name=None, **switches):
    """`window_metrics` per group of `key` — the one definition, applied group by group.

    `key` is a Series on `errors.index` (or an array of the same length). Rows whose key is
    missing are left out. Returns a frame indexed by (group, pair).
    """
    name = name or getattr(key, "name", None) or "group"
    key = pd.Series(np.asarray(key), index=errors.index, name=name)
    parts = {group: window_metrics(frame, **switches)
             for group, frame in errors.groupby(key, sort=True, observed=True)}
    return pd.concat(parts, names=[name, "pair"])


DISPLAY = {
    "MAE": "MAE [MWh]", "RMSE": "RMSE [MWh]", "bias": "bias [MWh]", "nMAE_pct": "nMAE [%]",
    "cap_MAE_pct": "cap. MAE [% of cap.]", "cap_bias_pct": "cap. bias [% of cap.]",
    "hour_count": "hour_count",
}


def show_metrics(table):
    """Display formatting only: labels, units, rounding; all-empty metric columns dropped."""
    out = table.dropna(axis=1, how="all").rename(index=PAIR_LABEL)
    out = out.round({"MAE": 0, "RMSE": 0, "bias": 0,
                     "nMAE_pct": 1, "cap_MAE_pct": 2, "cap_bias_pct": 2})
    return out.rename(columns=DISPLAY)


# Sanity check against a direct computation, so the helper is not trusted blindly.
_full = window_metrics(errors)
for actual in PAIRS:
    e = errors[ERR[actual]].dropna()
    assert np.isclose(_full.loc[actual, "MAE"], e.abs().mean())
    assert np.isclose(_full.loc[actual, "RMSE"], np.sqrt((e ** 2).mean()))
    assert _full.loc[actual, "hour_count"] == len(e)
print("window_metrics matches a direct computation on the full record")

### 3.3 Headline table

Every pair over two windows:

- **full record**: every pairwise-complete hour
- **trailing 365 days**: the 365 days ending at `JOINT_END`, the last hour at which all three pairs are observed (§2.2), derived from the data

These are the provisional benchmark numbers. The modelling spec re-scores SMARD on its own test
window from the hourly export (§8).

In [ ]:
TRAIL_START = JOINT_END - WINDOW + RESOLUTION   # exactly 365 days of hours, ending at JOINT_END

headline = pd.concat(
    {
        "full record": window_metrics(errors),
        "trailing 365 days": window_metrics(errors, start=TRAIL_START, end=JOINT_END),
    },
    names=["window"],
).swaplevel().reindex(pd.MultiIndex.from_product(
    [list(PAIRS), ["full record", "trailing 365 days"]], names=["pair", "window"]
))

print(f"full record      : {errors.index.min()} .. {errors.index.max()}")
print(f"trailing 365 days: {TRAIL_START} .. {JOINT_END}")
show_metrics(headline)

**SMARD misses residual load by ≈ 2,600–2,800 MWh per hour on average. The recent year is slightly worse and has flipped from under- to over-forecasting.**

- **Residual load: MAE 2,584 MWh over the full record, 2,790 MWh over the trailing 365 days.**
  - nMAE rises from 7.8 % to 10.1 %. Part of that is the smaller denominator (residual load keeps falling as wind + solar grow), not only a larger error. §4 separates the two with rolling lines.
  - RMSE (3,349 / 3,647 MWh) is ≈ 1.3× the MAE in both windows, so a few large misses carry noticeable weight.

- **The residual-load bias changed sign: −402 MWh (full record) → +414 MWh (trailing 365 days).**
  - The bias decomposes exactly (`bias_res = bias_load − bias_gen`): full record −494 − (−93) = −401; trailing 316 − (−98) = +414.
  - So the flip comes from **grid load**, which moved from under-forecast (−494 MWh) to over-forecast (+316 MWh). The wind + solar bias hardly moved (≈ −95 MWh in both windows).

- **Relative to its size, grid load is the most accurate pair.**
  - Grid load nMAE is 3.8 % in both windows. Wind + solar is at ≈ 7 %.
  - The wind + solar MAE grew from 1,505 to 1,782 MWh, but its nMAE stayed flat (7.0 % → 6.9 %). The larger absolute error tracks the larger generation, not a worse forecast. §4 checks this against installed capacity.

- **The load and generation errors are close to uncorrelated.**
  - Residual-load RMSE² ≈ load RMSE² + generation RMSE²: √(2,638² + 2,074²) ≈ 3,356 MWh against the measured 3,349 MWh over the full record, and 3,630 against 3,647 MWh over the trailing year. That only holds if the two component errors barely co-move. §5 reports the correlation per level bin.

- **`hour_count` is 8,759 for the trailing window, not 8,760:** 365 days × 24 h minus the missing 02:00 of the spring DST switch.

---

## 4 Time slices

The same metrics, sliced by time: month over the record, hour of day, season and year. Every
table comes from `window_metrics` / `metrics_by` (§3).

**Year-on-year statements come only from rolling 365-day lines.** Each point covers a full seasonal
cycle, so a partial year never needs a separate matched-window comparison and is never compared
directly against a full one.

### Explainer: what nMAE means for residual load

**nMAE (normalised MAE) = the average hourly miss in MWh ÷ the average residual load over the same hours.** It
answers one question: *how large is SMARD's typical error compared with the typical size of the
thing it forecasts?*

| Window | MAE | mean residual load | nMAE |
|---|---|---|---|
| first rolling year (window ending 2019-12) | 2,676 MWh | ≈ 37,900 MWh | **7.1 %** |
| last rolling year (window ending 2026-08) | 2,791 MWh | ≈ 27,900 MWh | **10.0 %** |

The forecast misses by about the same amount in MWh in both years (+4 %). But residual load itself
has shrunk by about a quarter, because wind and solar now cover more of the demand. The same miss is
therefore a bigger share of what's left: 10 % instead of 7 %. The rolling values come from §4.1 and
§4.5.

**How to read it**

- **It measures relative precision, not forecast skill.**
  - A rising nMAE does **not** mean SMARD got worse. Here it rises almost entirely because the denominator shrinks.
  - The components show that the forecasting itself did not degrade: grid-load nMAE is flat at ≈ 3.8 %, and wind + solar improves relative to installed capacity (§4.1).

- **It explains why residual load is the hardest pair in relative terms.**
  - Residual load is the difference of two large numbers: grid load (≈ 55,000 MWh on average, §3.3) minus wind + solar. Both errors carry over into the difference, but the difference is much smaller than either part.
  - So a grid-load error of ≈ 3.8 % becomes a residual-load error of ≈ 10 %.

- **It is what matters for this project's risk question.**
  - The low / negative risk direction is exactly the hours where residual load is near zero. A miss of ≈ 2,800 MWh is minor at 40,000 MWh, but at 0 MWh it decides whether an hour counts as oversupply at all.
  - The rising nMAE warns that the forecast's precision becomes more critical as residual load shrinks. §5 tests whether the error is actually worse in those hours.

**Where it breaks**

- **Short windows:** the ratio explodes when the mean gets close to zero. On 2025-10-26 the daily mean was 679 MWh, which would give an nMAE of ≈ 380 %. That's why nMAE is excluded at day level and in level bins (§3.1).

- **It is not an hourly percentage error.** That would be MAPE, which divides each hour's error by that hour's own value. MAPE is excluded because negative and near-zero hours make it meaningless.

- **Comparing years:** as residual load keeps falling, nMAE will keep rising even if SMARD's forecast stays exactly as good. So it can't be read as "better / worse over time" on its own, and §4.5 reads it together with the absolute MAE and the component lines.

**In short:** absolute MAE tells you *how many MWh SMARD misses by*. nMAE tells you *how much that
miss matters relative to the size of residual load*. For residual load the second number grows
because the target is shrinking, not because the forecast gets worse.

### Explainer: what bias means for residual load

**Bias = the average of the signed errors (`forecast − actual`) over a window.** Positive bias means
SMARD over-forecasts on average, negative means it under-forecasts. It answers one question: *does
the forecast lean systematically in one direction?* MAE answers a different one: *how far off is
it, in either direction?*

| Window | MAE | bias | reading |
|---|---|---|---|
| residual load, full record (§3.3) | 2,584 MWh | **−402 MWh** | misses are large but point both ways; a slight lean to under-forecasting |
| residual load, 2019 (§4.4) | 2,676 MWh | **−1,977 MWh** | most of the miss is one-directional: systematic under-forecasting |
| residual load, summer (§4.3) | 2,091 MWh | **+7 MWh** | practically no lean, but still ≈ 2,100 MWh off per hour |

**How to read it**

- **Bias is the systematic part of the error, MAE is the total.** MAE is always at least |bias|.
  - In 2019, |bias| is 74 % of MAE: SMARD was consistently too low, and grid load was under-forecast by −2,120 MWh that year.
  - Over the full record it is only 16 %: the errors mostly scatter around zero.
  - Put in RMSE terms: bias² is 35 % of the squared error in 2019 and only 1.4 % over the full record.

- **It decomposes exactly, which MAE does not.** `bias_res = bias_load − bias_gen`.
  - Full record: −494 − (−93) = −401 MWh. Trailing year: +316 − (−98) = +414 MWh (§3.3).
  - That is how this notebook can say the residual-load bias is a grid-load bias: the monthly correlation is 0.98 (§4.1).

- **The sign tells you which risk direction the forecast misses.**
  - **Negative bias (under-forecast):** actual residual load comes in higher than forecast, so **high-risk** hours are underestimated. Winter carries a bias of −1,200 MWh (§4.3), and winter is exactly when the high extremes occur.
  - **Positive bias (over-forecast):** actual residual load comes in lower than forecast, so **low / negative** hours are underestimated. The trailing year leans this way (+414 MWh).

- **A bias is correctable, scatter is not.**
  - At 07:00–08:00 residual load is under-forecast by 718–974 MWh on average (§4.2). A model that learns the offset by hour and season removes such a lean for free, so a systematic bias per slice is a direct hint for our own model's features.
  - Scatter around zero, like summer's +7 MWh bias against 2,091 MWh MAE, needs better information, not a correction.

**Where it breaks**

- **Positive and negative errors cancel.** A bias of zero does **not** mean an accurate forecast. At 13:00 the residual-load bias is −8 MWh, while the MAE there is ≈ 3,200 MWh, one of the worst hours of the day (§4.2).

- **Long windows hide systematic patterns.** Over the full record the bias is a modest −402 MWh, but it averages out opposite signs: −1,977 MWh in 2019, +1,005 MWh in 2022, −974 MWh at 08:00, +72 MWh at 12:00. Bias is therefore only informative per slice (hour, season, year), never as a single number.

- **Binning by the actual creates an artificial bias.** If you select the hours with the highest actual residual load, even a perfectly unbiased forecast looks like it under-forecasts there, because the selection picks the hours where reality overshot. This is the regression-to-the-mean effect that §5 handles by also binning by forecast level.

**In short:** MAE tells you *how far off* SMARD is. Bias tells you *which way it leans*, and in which
slices. For residual load the lean comes from grid load, changes sign between years and between
hours of the day, and cancels to a small number over the full record.

### 4.1 Month over the record, with rolling 365-day lines

- **Monthly MAE and bias** per pair. Only **complete** months are shown (`_complete_periods`), so an incomplete trailing month cannot appear as a fake dip or spike.
- **Rolling 365-day lines**, evaluated at each month end over the preceding 365 days:
  - rolling **MAE** per pair (MWh), which separates trend from seasonality
  - rolling **nMAE** per pair (%)
  - rolling **capacity-normalised MAE** for wind + solar (% of installed capacity)
- A rolling point exists only once the record spans a full 365 days before that month end. Earlier month ends stay empty rather than using a shorter window, which would cover only part of the seasonal cycle.

MWh and % never share an axis: the MWh lines are in the first figure, the normalised lines in the
second.

In [ ]:
MONTHS = _complete_periods(errors.index, "M")

# Month key: the month's start for complete months, NaT otherwise (dropped by metrics_by).
month_start = pd.Series(errors.index.to_period("M").start_time, index=errors.index)
month_start = month_start.where(errors.index.to_period("M").isin(MONTHS))
monthly = metrics_by(errors, month_start, name="month")

# Rolling lines: one full-window evaluation per complete month end.
MONTH_ENDS = MONTHS.end_time.floor("h")          # last hour of each complete month
rolling_parts = {}
for t in MONTH_ENDS:
    start = t - WINDOW + RESOLUTION
    if start < errors.index.min():                # no full 365 days yet: leave the point empty
        continue
    rolling_parts[t] = window_metrics(errors, start=start, end=t, capacity=True)
rolling = pd.concat(rolling_parts, names=["month_end", "pair"])

print(f"complete months            : {len(MONTHS)}  ({MONTHS.min()} .. {MONTHS.max()})")
print(f"rolling points (full 365 d): {len(rolling_parts)}  "
      f"(first month end {min(rolling_parts):%Y-%m-%d}, last {max(rolling_parts):%Y-%m-%d})")
print(f"month ends left empty      : {len(MONTH_ENDS) - len(rolling_parts)}")

In [ ]:
GRAY = "#707B8C"

fig, axes = plt.subplots(len(PAIRS), 1, figsize=(14, 12), sharex=True)
for ax, (actual, spec) in zip(axes, PAIRS.items()):
    m = monthly.xs(actual, level="pair")
    r = rolling.xs(actual, level="pair")
    ax.plot(m.index, m["MAE"], color=spec["color"], linewidth=1.2, alpha=0.55, label="monthly MAE")
    ax.plot(r.index, r["MAE"], color=spec["color"], linewidth=2.4, label="rolling 365-day MAE")
    ax.plot(m.index, m["bias"], color=GRAY, linewidth=1.2, linestyle="--", label="monthly bias")
    ax.axhline(0, color="0.6", linewidth=0.8)
    style_timeseries(ax, f"{spec['label']}: forecast error per month", "error [MWh]")
    ax.legend(loc="upper left", frameon=False, ncol=3)
fig.suptitle("SMARD day-ahead forecast error over the record (positive bias = over-forecast)",
             fontsize=16, y=1.0)
plt.tight_layout()
plt.show()

**The residual-load error comes in episodes, and its bias swings are grid-load swings.**

- **Monthly MAE spikes in a few episodes. The rolling 365-day MAE stays in a narrow band.**
  - Two months stand out: **2022-10 at 4,879 MWh** and **2021-01 at 4,660 MWh**, against a typical month of ≈ 2,000–3,000 MWh.
  - The rolling MAE stays between 2,109 MWh (window ending 2022-07) and 2,834 MWh (ending 2023-08) and ends at 2,791 MWh. The 2,109 MWh low is a window that happens to fall between the two episodes, which no calendar year shows (every full year is ≥ 2,259 MWh, §4.4).

- **Residual-load bias follows grid-load bias almost one to one.**
  - Across months the two biases correlate at **0.98**. In 2021-01 residual load was under-forecast by 4,538 MWh and grid load by 4,237 MWh; in 2022-10 residual load was over-forecast by 4,747 MWh and grid load by 4,682 MWh.
  - Grid load was under-forecast almost every month through 2019–2021. Since 2022 its bias swings around zero in both directions.

- **The wind + solar bias is small, but the MAE grows steadily.**
  - 84 % of months have a generation bias within ±500 MWh. The one large exception is 2026-05 (−1,119 MWh), which pushed residual load to +3,123 MWh that month together with +2,004 MWh from load.
  - The rolling generation MAE rises from 1,350 to 1,778 MWh. The next figure checks whether that is the forecast or the fleet.

In [ ]:
fig, (ax_n, ax_c) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for actual, spec in PAIRS.items():
    r = rolling.xs(actual, level="pair")
    ax_n.plot(r.index, r["nMAE_pct"], color=spec["color"], linewidth=2.2, label=spec["label"])
style_timeseries(ax_n, "Rolling 365-day nMAE per pair", "nMAE [%]")
ax_n.yaxis.set_major_formatter(lambda v, _: f"{v:.0f} %")
ax_n.legend(loc="upper left", frameon=False, ncol=3)

r = rolling.xs("renewables", level="pair")
ax_c.plot(r.index, r["cap_MAE_pct"], color=PAIRS["renewables"]["color"], linewidth=2.2,
          label="Wind + solar")
style_timeseries(ax_c, "Rolling 365-day wind + solar MAE, normalised by installed capacity",
                 "MAE [% of installed capacity]")
ax_c.yaxis.set_major_formatter(lambda v, _: f"{v:.2f} %")
ax_c.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

**Normalised, grid load and wind + solar are stable or better. Only residual load's nMAE rises, because its mean is shrinking.**

- **The wind + solar forecast improves relative to the fleet it forecasts.**
  - Rolling capacity-normalised MAE falls from **1.31 %** (window ending 2019-12) to **1.03 %** (2026-08). The absolute MAE rose from 1,350 to 1,778 MWh over the same windows, so the rise comes from the larger fleet.
  - Its rolling nMAE stays in a band of ≈ 6.5–7.8 % throughout.

- **Grid load has no clear direction.**
  - Rolling nMAE moves between 3.0 % and 4.3 % and ends at 3.8 %, close to where it started (4.2 %).

- **Residual-load nMAE climbs from 7.1 % to 10.0 %, but mostly through its denominator.**
  - Mean residual load fell from ≈ 37,900 MWh (the first rolling window) to ≈ 27,900 MWh (the last), while the absolute rolling MAE rose only from 2,676 to 2,791 MWh (+4 %).
  - The same error in MWh is therefore a larger share of a smaller residual load. §4.5 reads the two lines together.

### 4.2 Hour of day

MAE and bias per pair by local hour (0–23). There is no nMAE here (§3.1). The hour-02 row has one
observation fewer per year: the missing spring-DST hour, visible in `hour_count`.

In [ ]:
by_hour = metrics_by(errors, time_series["hour"], name="hour", nmae=False)

fig, (ax_mae, ax_bias) = plt.subplots(1, 2, figsize=(15, 4.8), sharex=True)
for actual, spec in PAIRS.items():
    h = by_hour.xs(actual, level="pair")
    ax_mae.plot(h.index, h["MAE"], color=spec["color"], linewidth=2, marker="o", markersize=4,
                label=spec["label"])
    ax_bias.plot(h.index, h["bias"], color=spec["color"], linewidth=2, marker="o", markersize=4,
                 label=spec["label"])
ax_bias.axhline(0, color="0.6", linewidth=0.8)
for ax, title, ylabel in [(ax_mae, "MAE by hour of day", "MAE [MWh]"),
                          (ax_bias, "Bias by hour of day (positive = over-forecast)", "bias [MWh]")]:
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("hour of day (local time)")
    ax.set_ylabel(ylabel, color="grey")
    ax.set_xticks(range(0, 24, 2))
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax_mae.legend(frameon=False)
plt.tight_layout()
plt.show()

# The weakest hours per pair, named explicitly.
for actual, spec in PAIRS.items():
    h = by_hour.xs(actual, level="pair")
    worst = h["MAE"].nlargest(3)
    best = h["MAE"].idxmin()
    print(f"{spec['label']:14s} weakest hours: "
          + ", ".join(f"{hr:02d}:00 ({v:,.0f} MWh)" for hr, v in worst.items())
          + f"   |  best: {best:02d}:00 ({h.loc[best, 'MAE']:,.0f} MWh)")
print(f"\nhour_count at 02:00: {by_hour.xs('grid_load', level='pair').loc[2, 'hour_count']:,} "
      f"vs 03:00: {by_hour.xs('grid_load', level='pair').loc[3, 'hour_count']:,}")

**All three pairs are weakest around midday. The generation error peaks with the sun, as expected.**

- **Weakest hours: 12:00–14:00 for every pair.**
  - Wind + solar: 2,097 MWh at 13:00 against 1,077 MWh at 00:00, almost double. This is the solar peak.
  - Residual load: 3,212 MWh at 12:00 against 2,104 MWh at 00:00.
  - Grid load also peaks at midday (2,355 MWh at 13:00), but less sharply (best: 1,821 MWh at 04:00).

- **Grid load is under-forecast around the clock except at midday.**
  - The bias is most negative in the evening (−952 MWh at 19:00) and the morning (−902 MWh at 07:00). It turns positive only at 11:00–15:00 (+237 MWh at 13:00).

- **At midday the two component biases cancel in residual load. In the morning they add up.**
  - At 13:00 load is over-forecast by +237 MWh and generation by +245 MWh. Since `bias_res = bias_load − bias_gen`, residual load ends up at −8 MWh.
  - At 08:00 residual load has its largest bias (−974 MWh), almost all from load (−892 MWh).

- **DST shows up as expected:** `hour_count` is 2,797 at 02:00 against 2,805 at 03:00, one missing hour per spring switch.

### 4.3 Season

Meteorological seasons (the inherited mapping), pooled over the whole record. Each season is
covered a different number of times: the partial final year adds some seasons but not others, as
`hour_count` shows.

In [ ]:
by_season = metrics_by(errors, time_series["season"], name="season")
by_season = by_season.reindex(pd.MultiIndex.from_product([SEASON_ORDER, list(PAIRS)],
                                                         names=["season", "pair"]))
show_metrics(by_season)

**Winter is the hardest season and summer the easiest. The seasonal bias is again a load bias.**

- **Residual-load MAE: 2,972 MWh in winter, 2,091 MWh in summer.**
  - The winter bias of −1,200 MWh is almost entirely grid load (−1,312 MWh). Autumn looks the same (−677 vs −739 MWh). Wind + solar bias stays between −212 and +18 MWh in every season.
  - Summer is nearly unbiased for all three pairs (≤ 24 MWh).

- **Residual-load nMAE is highest in spring (8.9 %),** although winter has the larger MAE. Spring's lower mean residual load raises the ratio.

- **The seasons are not equally covered:** autumn has 15,432 hours against 17,664 in summer, because the partial final year has no autumn yet.

### 4.4 Year

Two tables:

1. **MAE, RMSE, bias and nMAE per pair and year**
2. **the capacity-normalised error for wind + solar per year.** It exists only for the generation pair (§3.1), so it has its own table instead of empty cells for the other two pairs.

**Partial years (first or last) are marked** and are not compared against full years. They are
listed for completeness, and the rolling lines in §4.1 carry every year-on-year statement.

In [ ]:
COMPLETE_YEARS = {int(p.year) for p in _complete_periods(errors.index, "Y")}

by_year = metrics_by(errors, time_series["year"], name="year", capacity=True)
CAP_COLUMNS = ["cap_MAE_pct", "cap_bias_pct"]


def partial_marker(years):
    return ["partial" if int(y) not in COMPLETE_YEARS else "" for y in years]


# Table 1: all pairs, without the capacity columns (they apply to wind + solar only).
year_table = show_metrics(by_year.drop(columns=CAP_COLUMNS))
year_table.insert(0, "partial year", partial_marker(by_year.index.get_level_values("year")))

# Table 2: wind + solar only, capacity-normalised error per year.
cap_year = by_year.xs("renewables", level="pair")[CAP_COLUMNS + ["hour_count"]]
cap_table = show_metrics(cap_year)
cap_table.insert(0, "partial year", partial_marker(cap_year.index))

print(f"complete years: {sorted(COMPLETE_YEARS)}")
print(f"partial years : {[y for y in YEARS if y not in COMPLETE_YEARS]}")
print("\nMAE, RMSE, bias and nMAE per pair and year:")
display(year_table)
print("Wind + solar error, normalised by installed capacity, per year:")
display(cap_table)

**No full year stands out for residual load. The bias changes sign from year to year, following grid load.**

- **Residual-load MAE ranges from 2,259 MWh (2020) to 2,676 MWh (2019) across the seven full years,** with no ordering by time.
  - The bias sign changes: −1,977 / −1,138 / −1,788 MWh in 2019–2021, +1,005 / +265 in 2022–2023, −523 in 2024, +620 in 2025. Each year's grid-load bias has the same sign and a similar size.

- **The capacity-normalised wind + solar MAE falls from 1.31 % (2019) to 1.07 % (2025),** the same improvement as the rolling line.

- **2026 is partial (5,975 hours) and is not compared.** Its residual-load MAE of 2,964 MWh covers only January to early September, a different mix of seasons from a full year.

- **`hour_count` is 8,759 in a normal year and 8,783 in a leap year** (365 or 366 days × 24 h, minus the missing spring-DST hour). In 2020 residual load and grid load show 8,759 only because the 24 hours of 2020-01-31 have no forecast (§2.2).

### 4.5 Is the forecast getting better or worse?

The verdict per pair comes from the **rolling normalised lines** (§4.1), not from the year table:

- **grid load**: rolling nMAE
- **wind + solar**: rolling capacity-normalised MAE, because absolute generation error grows with the fleet
- **residual load**: rolling MAE and rolling nMAE, read together with the decomposition. A rise in absolute residual-load MAE is not by itself evidence of a worse forecast: it can come entirely from a larger generation fleet.

The table lists each line's first and last point and its range. The record holds only a few full
rolling years, which is not enough for a precise trend, so none is claimed.

In [ ]:
VERDICT_LINES = [
    ("residual_load", "MAE", "MWh"),
    ("residual_load", "nMAE_pct", "%"),
    ("grid_load", "nMAE_pct", "%"),
    ("renewables", "cap_MAE_pct", "% of cap."),
    ("renewables", "MAE", "MWh"),
]

rows = []
for actual, metric, unit in VERDICT_LINES:
    line = rolling.xs(actual, level="pair")[metric]
    rows.append({
        "pair": PAIR_LABEL[actual],
        "rolling metric": f"{DISPLAY[metric].split(' [')[0]} [{unit}]",
        "first": line.iloc[0],
        "last": line.iloc[-1],
        "change": line.iloc[-1] - line.iloc[0],
        "min": line.min(),
        "min at": f"{line.idxmin():%Y-%m}",
        "max": line.max(),
        "max at": f"{line.idxmax():%Y-%m}",
    })

verdict = pd.DataFrame(rows).set_index(["pair", "rolling metric"])
print(f"first rolling point: window ending {rolling.index.get_level_values('month_end').min():%Y-%m-%d}; "
      f"last: window ending {rolling.index.get_level_values('month_end').max():%Y-%m-%d}")
verdict.round(2)

**Verdict per pair, from the rolling normalised lines. The direction is stated, not a precise trend.**

- **Wind + solar: improving.**
  - Rolling capacity-normalised MAE fell from 1.31 % to 1.03 % of installed capacity, and the line reaches its minimum (1.02 %) in the last year of the record.

- **Grid load: no clear direction.**
  - Rolling nMAE started at 4.23 % and ended at 3.84 %, but it moved between 3.04 % and 4.28 % in between. The ending is not outside the earlier range.

- **Residual load: about stable in MWh, worse relative to its shrinking size.**
  - The absolute rolling MAE went from 2,676 to 2,791 MWh (+115 MWh, +4 %), within its earlier range (max 2,834 MWh).
  - The rolling nMAE went from 7.07 % to 9.99 %, its maximum. That comes mostly from mean residual load falling by roughly a quarter, not from a larger error: both components are stable or better (above).
  - For this project the relative view matters: the same ≈ 2,800 MWh miss now falls on a residual load that more often sits near zero. §5 tests whether the error is worse exactly there, in the tails.